# Лабораторная работа 5. Классификация слов предложений на основе нейронной сети

Лабораторная работа составлена на основе [материала](https://colab.research.google.com/drive/1Pz8b_h-W9zIBk1p2e6v-YFYThG1NkYeS?usp=sharing) о PyTorch из [курса ](https://web.stanford.edu/class/cs224n/).

#### Работу выполнил: {впишите вашу фамилию} {впишите ваше имя}, студент группы ФИб-4

Рассмотрим применение библиотеки PyTorch дл решения NLP задачи - классификация слов на класс именованной сущности Location (задача Name Entity Recognition, NER). Познакомимся со следующими этапами:

* Подготовка данных
* Моделирование
* Обучение
* Оценивание качества

**Постановка задачи**: нужно обучить модель  определять относится ли слово (с учётом его N-соседей в предложении) к названию местоположения (`LOCATION`).\
(!) Решаем задачу в условиях ограничения, что названия местоположений `LOCATION`, состоящих из нескольких слов, не учитываются.

In [ ]:
import torch
from torch import nn

Исходный корпус:

In [ ]:
corpus = [
          "We always come to Paris",
          "The professor is from Australia",
          "I live in Stanford",
          "He comes from Taiwan",
          "The capital of Turkey is Ankara"
         ]

Обучающие предложения разбьем по словам.

In [ ]:
def preprocess_sentence(sentence):
  return sentence.lower().split()

train_sentences = [preprocess_sentence(sent) for sent in corpus]
train_sentences

[['we', 'always', 'come', 'to', 'paris'],
 ['the', 'professor', 'is', 'from', 'australia'],
 ['i', 'live', 'in', 'stanford'],
 ['he', 'comes', 'from', 'taiwan'],
 ['the', 'capital', 'of', 'turkey', 'is', 'ankara']]

Определим соответствующие метки (относится/не относится  слово к `LOCATION`) для слов обучающих предложений

In [ ]:
locations = set(["australia", "ankara", "paris", "stanford", "taiwan", "turkey"])

train_labels = [[1 if word in locations else 0 for word in sent] for sent in train_sentences]
train_labels

[[0, 0, 0, 0, 1],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 1],
 [0, 0, 0, 1],
 [0, 0, 0, 1, 0, 1]]

Составим словарь нашего корпуса

In [ ]:
vocabulary = set(w for s in train_sentences for w in s)
vocabulary

{'always',
 'ankara',
 'australia',
 'capital',
 'come',
 'comes',
 'from',
 'he',
 'i',
 'in',
 'is',
 'live',
 'of',
 'paris',
 'professor',
 'stanford',
 'taiwan',
 'the',
 'to',
 'turkey',
 'we'}

Добавим специальный токен `<unk>` в словарь `vocabular`, который будет соответсвовать словам не из словаря.

In [ ]:
vocabulary.add("<unk>")

Модель, которую будем обучать, будет смотреть не на всё предложение или на текст, а на окно слов (Word Window) в предложении. Окном яляется последовательность из слова и его N-соседей слева и справа.

Например:\
при N=2 для слова `Turkey` в предложении `The capital of Turkey is Ankara` окно будет `capital of Turkey is Ankara`.

Чтобы для слов, стоящих в начале и конце предложения, находились слова-соседи добавим в словарь `vocabular` токен `<pad>`.

In [ ]:
vocabulary.add("<pad>")

In [ ]:
def pad_window(sentence, window_size, pad_token="<pad>"):
  window = [pad_token] * window_size
  return window + sentence + window

window_size = 2
pad_window(train_sentences[0], window_size=window_size)

['<pad>', '<pad>', 'we', 'always', 'come', 'to', 'paris', '<pad>', '<pad>']

Отсортируем словарь и назначим каждому токену (слову) его индекс.

In [ ]:
ix_to_word = sorted(list(vocabulary))

word_to_ix = {word: ind for ind, word in enumerate(ix_to_word)}
word_to_ix

{'<pad>': 0,
 '<unk>': 1,
 'always': 2,
 'ankara': 3,
 'australia': 4,
 'capital': 5,
 'come': 6,
 'comes': 7,
 'from': 8,
 'he': 9,
 'i': 10,
 'in': 11,
 'is': 12,
 'live': 13,
 'of': 14,
 'paris': 15,
 'professor': 16,
 'stanford': 17,
 'taiwan': 18,
 'the': 19,
 'to': 20,
 'turkey': 21,
 'we': 22}

In [ ]:
ix_to_word[1]

'<unk>'

Теперь можно преобразовать обучающие предложения в последовательность индексов, соответствующих каждому токену.

In [ ]:
def convert_token_to_indices(sentence, word_to_ix):
  indices = []
  for token in sentence:
    if token in word_to_ix:
      index = word_to_ix[token]
    else:
      index = word_to_ix["<unk>"]
    indices.append(index)
  return indices

Эта же функция преобразования предложения в последовательность индекстов словаря токенов, но в компактном виде

In [ ]:
def _convert_token_to_indices(sentence, word_to_ix):
  return [word_to_ind.get(token, word_to_ix["<unk>"]) for token in sentence]

Пример преобразования список слов в список соответсвующих индексов и обратно.

In [ ]:
example_sentence = ["we", "always", "come", "to", "kuwait"]
example_indices = convert_token_to_indices(example_sentence, word_to_ix)
restored_example = [ix_to_word[ind] for ind in example_indices]

print(f"Original sentence is: {example_sentence}")
print(f"Going from words to indices: {example_indices}")
print(f"Going from indices to words: {restored_example}")

Original sentence is: ['we', 'always', 'come', 'to', 'kuwait']
Going from words to indices: [22, 2, 6, 20, 1]
Going from indices to words: ['we', 'always', 'come', 'to', '<unk>']


Построим для каждого обучающего предложения массив из соответсвующих индексов.

In [ ]:
example_padded_indices = [convert_token_to_indices(s, word_to_ix) for s in train_sentences]
example_padded_indices

[[22, 2, 6, 20, 15],
 [19, 16, 12, 8, 4],
 [10, 13, 11, 17],
 [9, 7, 8, 18],
 [19, 5, 14, 21, 12, 3]]

Создадим таблицу эмбеддингов (для представления слов в виде векторов), которую представим в виде специального слоя `nn.Embedding(num_words, embedding_dimension)`, где `num_words` - количество слов в словаре, `embedding_dimension` - размерность вектора представления. Начальные значения таблицы выбираются случайным образом. Параметры этого слоя будут меняться по мере обучения нейронной сети на решаемой задаче и векторные представления будут улучшаться относительно решаемой задачи.

In [ ]:
embedding_dim = 5
embeds = nn.Embedding(len(vocabulary), embedding_dim)

list(embeds.named_parameters())

[('weight',
  Parameter containing:
  tensor([[ 3.5615e-01, -7.6074e-01,  1.2887e+00, -4.3196e-02,  2.1726e-01],
          [-6.7596e-01,  3.5717e-01, -8.8961e-01,  8.5136e-01, -8.7651e-01],
          [ 5.6443e-01,  1.0828e+00, -1.0699e+00, -1.1550e+00,  5.0433e-01],
          [-5.3939e-01, -1.5750e+00, -5.1491e-01, -4.8407e-01,  5.3563e-01],
          [-1.8745e+00,  5.6361e-01, -6.0366e-01,  7.1338e-01, -3.7444e-01],
          [ 1.9365e+00,  9.1059e-01, -5.2000e-01, -1.0364e+00,  2.7182e-01],
          [ 1.4048e+00, -1.4530e-01, -1.7402e+00, -1.2060e-01,  1.2157e+00],
          [-2.0097e-01,  3.7349e-01, -7.9907e-01, -9.1029e-01,  6.8405e-01],
          [-2.1283e-02, -4.3873e-01, -1.0316e+00,  1.4856e+00,  1.5170e-01],
          [-1.0443e+00,  6.4177e-01, -1.5370e-01, -4.1663e-01, -2.4606e-03],
          [-4.5848e-02, -2.8905e-01, -1.2548e+00, -2.5571e-01, -2.1269e-01],
          [ 4.5490e-01, -4.8078e-01,  1.8342e+00,  6.0927e-01, -1.2865e+00],
          [ 2.3220e-01, -1.5592e+00, -2.

Пример векторного представления для слова `paris`.

In [ ]:
index = word_to_ix["paris"]
print(f"Индекс для слова `paris`: {index}")
index_tensor = torch.tensor(index, dtype=torch.long)
print(f"Тензор для индекса: {index_tensor}")
paris_embed = embeds(index_tensor)
print(f"Векторное представление для слова `paris`: {paris_embed}")
paris_embed

Индекс для слова `paris`: 15
Тензор для индекса: 15
Векторное представление для слова `paris`: tensor([ 0.1093,  0.7146, -0.4913,  3.1117, -1.3227],
       grad_fn=<EmbeddingBackward0>)


tensor([ 0.1093,  0.7146, -0.4913,  3.1117, -1.3227],
       grad_fn=<EmbeddingBackward0>)

Можно получать векторые представления для нескольких слов (предложения, текста)

In [ ]:
index_paris = word_to_ix["paris"]
index_ankara = word_to_ix["ankara"]
indices = [index_paris, index_ankara]
indices_tensor = torch.tensor(indices, dtype=torch.long)
embeddings = embeds(indices_tensor)
embeddings

tensor([[ 0.1093,  0.7146, -0.4913,  3.1117, -1.3227],
        [ 0.7202, -0.5314, -0.0293, -1.2950,  0.4904]],
       grad_fn=<EmbeddingBackward0>)

Обычно слой эмбеддингов определяют как часть обучаемой модели.

Загрузка данных в модель обычно происходит батчами (пакетами). Для этого используется специальный класс `torch.util.data.DataLoader`.

Будем вызывать данный класс как `DataLoader(data, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)`. Аргумент `batch_size` - определяет размер порции данных, которая будет отсылаться за один шаг. Аргумент `shuffle` отвечает за перемешивание данных, а `collate_fn` - определяет функцию для дополнительной обработки батча.

In [ ]:
from torch.utils.data import DataLoader
from functools import partial

def custom_collate_fn(batch, window_size, word_to_ix):

  def pad_window(sentence, window_size, pad_token="<pad>"):
    window = [pad_token] * window_size
    return window + sentence + window

  def convert_tokens_to_indices(sentence, word_to_ix):
    return [word_to_ix.get(token, word_to_ix["<unk>"]) for token in sentence]

  # Prepare the datapoints
  x, y = zip(*batch)
  x = [pad_window(s, window_size=window_size) for s in x]
  x = [convert_tokens_to_indices(s, word_to_ix) for s in x]

  # Pad x so that all the examples in the batch have the same size
  pad_token_ix = word_to_ix["<pad>"]
  x = [torch.LongTensor(x_i) for x_i in x]
  x_padded = nn.utils.rnn.pad_sequence(x, batch_first=True, padding_value=pad_token_ix)

  # Pad y and record the length
  lengths = [len(label) for label in y]
  lenghts = torch.LongTensor(lengths)
  y = [torch.LongTensor(y_i) for y_i in y]
  y_padded = nn.utils.rnn.pad_sequence(y, batch_first=True, padding_value=0)

  return x_padded, y_padded, lenghts

Запустим `DataLoader` и посмотрим данные для обучения.

In [ ]:
data = list(zip(train_sentences, train_labels))
batch_size = 2
shuffle = True
window_size = 2
collate_fn = partial(custom_collate_fn, window_size=window_size, word_to_ix=word_to_ix)

loader = DataLoader(data, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn)

counter = 0
for batched_x, batched_y, batched_lengths in loader:
  print(f"Iteration {counter}")
  print("Batched Input:")
  print(batched_x)
  print("Batched Labels:")
  print(batched_y)
  print("Batched Lengths:")
  print(batched_lengths)
  print("")
  counter += 1

Iteration 0
Batched Input:
tensor([[ 0,  0, 19, 16, 12,  8,  4,  0,  0],
        [ 0,  0, 22,  2,  6, 20, 15,  0,  0]])
Batched Labels:
tensor([[0, 0, 0, 0, 1],
        [0, 0, 0, 0, 1]])
Batched Lengths:
tensor([5, 5])

Iteration 1
Batched Input:
tensor([[ 0,  0, 19,  5, 14, 21, 12,  3,  0,  0],
        [ 0,  0,  9,  7,  8, 18,  0,  0,  0,  0]])
Batched Labels:
tensor([[0, 0, 0, 1, 0, 1],
        [0, 0, 0, 1, 0, 0]])
Batched Lengths:
tensor([6, 4])

Iteration 2
Batched Input:
tensor([[ 0,  0, 10, 13, 11, 17,  0,  0]])
Batched Labels:
tensor([[0, 0, 0, 1]])
Batched Lengths:
tensor([4])



Батчи, которые вывелись выше будут передаваться в модель. По заданию модель должна смотреть на текстовое окно и классифицировать его, относится ли центральное слово окна к `LOCATION`. Для этого из предложений нужно выделять данные окна.

In [ ]:
# Исходный тензор
print(f"Original Tensor: ")
print(batched_x)
print("")

# Создание окон из предложения
chunk = batched_x.unfold(1, window_size*2 + 1, 1)
print(f"Windows: ")
print(chunk)

Original Tensor: 
tensor([[ 0,  0, 10, 13, 11, 17,  0,  0]])

Windows: 
tensor([[[ 0,  0, 10, 13, 11],
         [ 0, 10, 13, 11, 17],
         [10, 13, 11, 17,  0],
         [13, 11, 17,  0,  0]]])


Теперь создадим модель нейронной сети.

In [ ]:
class WordWindowClassifier(nn.Module):

  def __init__(self, hyperparameters, vocab_size, pad_ix=0):
    super(WordWindowClassifier, self).__init__()

    """ Instance variables """
    self.window_size = hyperparameters["window_size"]
    self.embed_dim = hyperparameters["embed_dim"]
    self.hidden_dim = hyperparameters["hidden_dim"]
    self.freeze_embeddings = hyperparameters["freeze_embeddings"]

    """ Embedding Layer
    Takes in a tensor containing embedding indices, and returns the
    corresponding embeddings. The output is of dim
    (number_of_indices * embedding_dim).

    If freeze_embeddings is True, set the embedding layer parameters to be
    non-trainable. This is useful if we only want the parameters other than the
    embeddings parameters to change.

    """
    self.embeds = nn.Embedding(vocab_size, self.embed_dim, padding_idx=pad_ix)
    if self.freeze_embeddings:
      self.embed_layer.weight.requires_grad = False

    """ Hidden Layer
    """
    full_window_size = 2 * window_size + 1
    self.hidden_layer = nn.Sequential(
      nn.Linear(full_window_size * self.embed_dim, self.hidden_dim),
      nn.Tanh()
    )

    """ Output Layer
    """
    self.output_layer = nn.Linear(self.hidden_dim, 1)

    """ Probabilities
    """
    self.probabilities = nn.Sigmoid()

  def forward(self, inputs):
    """
    Let B:= batch_size
        L:= window-padded sentence length
        D:= self.embed_dim
        S:= self.window_size
        H:= self.hidden_dim

    inputs: a (B, L) tensor of token indices
    """
    B, L = inputs.size()

    """
    Reshaping.
    Takes in a (B, L) LongTensor
    Outputs a (B, L~, S) LongTensor
    """
    # Fist, get our word windows for each word in our input.
    token_windows = inputs.unfold(1, 2 * self.window_size + 1, 1)
    _, adjusted_length, _ = token_windows.size()

    # Good idea to do internal tensor-size sanity checks, at the least in comments!
    assert token_windows.size() == (B, adjusted_length, 2 * self.window_size + 1)

    """
    Embedding.
    Takes in a torch.LongTensor of size (B, L~, S)
    Outputs a (B, L~, S, D) FloatTensor.
    """
    embedded_windows = self.embeds(token_windows)

    """
    Reshaping.
    Takes in a (B, L~, S, D) FloatTensor.
    Resizes it into a (B, L~, S*D) FloatTensor.
    -1 argument "infers" what the last dimension should be based on leftover axes.
    """
    embedded_windows = embedded_windows.view(B, adjusted_length, -1)

    """
    Layer 1.
    Takes in a (B, L~, S*D) FloatTensor.
    Resizes it into a (B, L~, H) FloatTensor
    """
    layer_1 = self.hidden_layer(embedded_windows)

    """
    Layer 2
    Takes in a (B, L~, H) FloatTensor.
    Resizes it into a (B, L~, 1) FloatTensor.
    """
    output = self.output_layer(layer_1)

    """
    Softmax.
    Takes in a (B, L~, 1) FloatTensor of unnormalized class scores.
    Outputs a (B, L~, 1) FloatTensor of (log-)normalized class scores.
    """
    output = self.probabilities(output)
    output = output.view(B, -1)

    return output

Выполним подготовку для обучения модели.

In [ ]:
data = list(zip(train_sentences, train_labels))
batch_size = 2
shuffle = True
window_size = 2
collate_fn = partial(custom_collate_fn, window_size=window_size, word_to_ix=word_to_ix)

loader = DataLoader(data, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn)

model_hyperparameters = {
    "batch_size": 4,
    "window_size": 2,
    "embed_dim": 25,
    "hidden_dim": 25,
    "freeze_embeddings": False,
}

vocab_size = len(word_to_ix)
model = WordWindowClassifier(model_hyperparameters, vocab_size)

learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# Определим функцию потерь
def loss_function(batch_outputs, batch_labels, batch_lengths):
    # Вычисление ошибки по всему батчу
    bceloss = nn.BCELoss()
    loss = bceloss(batch_outputs, batch_labels.float())

    # Масштабирование ошибки относительно количества слов в батче
    loss = loss / batch_lengths.sum().float()

    return loss

Определим функцию обучения и функцию обучения на одной эпохе.

In [ ]:
def train_epoch(loss_function, optimizer, model, loader):
  total_loss = 0
  for batch_inputs, batch_labels, batch_lengths in loader:
    # Clear the gradients
    optimizer.zero_grad()
    # Run a forward pass
    outputs = model.forward(batch_inputs)
    # Compute the batch loss
    loss = loss_function(outputs, batch_labels, batch_lengths)
    # Calculate the gradients
    loss.backward()
    # Update the parameteres
    optimizer.step()
    total_loss += loss.item()

  return total_loss


def train(loss_function, optimizer, model, loader, num_epochs=10000):
  for epoch in range(num_epochs):
    epoch_loss = train_epoch(loss_function, optimizer, model, loader)
    if epoch % 100 == 0: print(f"Value loss on {epoch} is {epoch_loss}")

Запустим обучение модели.

In [ ]:
num_epochs = 1000
train(loss_function, optimizer, model, loader, num_epochs=num_epochs)

Value loss on 0 is 0.327986940741539
Value loss on 100 is 0.25638044998049736
Value loss on 200 is 0.1803244799375534
Value loss on 300 is 0.15306463092565536
Value loss on 400 is 0.10852054134011269
Value loss on 500 is 0.09015713445842266
Value loss on 600 is 0.06988732144236565
Value loss on 700 is 0.05571055132895708
Value loss on 800 is 0.05116669926792383
Value loss on 900 is 0.03757527191191912


Теперь для проверки обученой модели нужны тестовые данные.

In [ ]:
# Create test sentences
test_corpus = ["I was born in Kirov", "Paris is a capital of France"]
test_sentences = [s.lower().split() for s in test_corpus]
test_labels = [[0, 0, 0, 0, 1], [1, 0, 0, 0, 0, 1]]

# Create a test loader
test_data = list(zip(test_sentences, test_labels))
batch_size = 1
shuffle = False
window_size = 2
collate_fn = partial(custom_collate_fn, window_size=2, word_to_ix=word_to_ix)
test_loader = torch.utils.data.DataLoader(test_data,
                                           batch_size=1,
                                           shuffle=False,
                                           collate_fn=collate_fn)

In [ ]:
for x in test_loader:
  print(x)

(tensor([[ 0,  0, 10,  1,  1, 11,  1,  0,  0]]), tensor([[0, 0, 0, 0, 1]]), tensor([5]))
(tensor([[ 0,  0, 15, 12,  1,  5, 14,  1,  0,  0]]), tensor([[1, 0, 0, 0, 0, 1]]), tensor([6]))


In [ ]:
for test_instance, labels, _ in test_loader:
  outputs = model.forward(test_instance)
  print(f"Истинные метки: {labels}")
  print(f"Предсказанные p(w is `LOCATION`):\n{outputs}")
  print('*'*32)

Истинные метки: tensor([[0, 0, 0, 0, 1]])
Предсказанные p(w is `LOCATION`):
tensor([[0.1281, 0.0809, 0.1302, 0.1097, 0.5995]], grad_fn=<ViewBackward0>)
********************************
Истинные метки: tensor([[1, 0, 0, 0, 0, 1]])
Предсказанные p(w is `LOCATION`):
tensor([[0.6966, 0.1081, 0.3965, 0.0579, 0.0636, 0.4891]],
       grad_fn=<ViewBackward0>)
********************************


**Задание 1.** Проанализируйте результаты предсказаний обученной модели на новых данных. В каких случаях модель более уверена в своих ответах?

**Задание 2.** Меняя размерность векторных представлений (эмбеддингов) у модели (нужно будет создать и обучить несколько экземпляров класса модели), проанализируйте качество обучения в случае когда векторные представления имеют меньшую и большую размерность, чем в текущем руководстве (например, 5 и 50). Визуализируйте результаты.

**Задание 3.** Подготовьте данные из 25 предложений на русском языке с наличием слов, относящихся к именнованной сущности `ORGANIZATION` (20 для обучения и 5 для тестирования). При составлении предложений можно использовать сгенерированные вручную тексты. Автоматизируйте процесс создания корпуса, а точнее его разметки, например можно задать словарь организаций, а метки класса расставлять в цикле при проходе по словам, выполняя проверку входит ли начальная форма слова в словарь организаций.\
Создайте и обучите модель задаче классификации слова на класс именованной сущности `ORGANIZATION`. Подберите наиболее оптимальные гиперпараметры обучения. Визуализируйте процесс обучения (постройте графики loss). Проанализируйте качество предсказаний модели и сделайте выводы.